In [ ]:
%pip install -U langchain-tavily


In [46]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.prebuilt import ToolNode


import os
from dotenv import load_dotenv

load_dotenv()


True

In [47]:
@tool
def search_docs(name: str) -> str:
    """
    知识库工具，模拟内部知识
    """
    dic = {
        "李小明": "性别：男，年龄：25，手机号：13812345678，邮箱：lism@example.com，工作：软件工程师，公司：XX科技有限公司，部门：开发部，岗位：前端开发工程师，职级：中级，薪资：15000元/月，入职时间：2022-03-15，学历：本科（计算机科学与技术），技能：Vue3、TypeScript、Element Plus、工程化部署，直属领导：张技术主管，团队：用户端开发组，福利：五险一金+年终奖（1-3个月薪资）+ 年度调薪",
        "王小红": "性别：女，年龄：28，手机号：13987654321，邮箱：wxh@example.com，工作：数据分析师，公司：XX数据分析有限公司，部门：数据分析部，岗位：数据分析师，职级：高级，薪资：18000元/月，入职时间：2020-09-01，学历：硕士（应用统计学），技能：SQL、Python（Pandas/Numpy）、Tableau可视化、用户行为分析，直属领导：刘数据总监，团队：商业分析组，福利：五险一金+补充医疗+年度体检+带薪年假15天",
        "张建国": "性别：男，年龄：32，手机号：13711223344，邮箱：zjg@example.com，工作：软件工程师，公司：XX科技有限公司，部门：开发部，岗位：后端开发工程师，职级：高级，薪资：25000元/月，入职时间：2019-07-20，学历：本科（软件工程），技能：Java、SpringCloud、MySQL、Redis、微服务架构，直属领导：张技术主管，团队：服务端开发组，福利：五险一金+股票期权+年度团建+免费技术培训",
        "刘婷婷": "性别：女，年龄：24，手机号：13655667788，邮箱：lt@example.com，工作：软件工程师，公司：XX科技有限公司，部门：开发部，岗位：测试开发工程师，职级：初级，薪资：12000元/月，入职时间：2023-06-01，学历：本科（自动化），技能：Python、Selenium、Jest、接口测试，直属领导：张技术主管，团队：质量保障组，福利：五险一金+试用期全额薪资+导师带教",
        "陈浩然": "性别：男，年龄：29，手机号：13599887766，邮箱：chr@example.com，工作：软件工程师，公司：XX科技有限公司，部门：开发部，岗位：算法工程师，职级：中级，薪资：30000元/月，入职时间：2021-01-10，学历：硕士（人工智能），技能：机器学习、TensorFlow、NLP、推荐系统，直属领导：赵算法总监，团队：算法研发组，福利：五险一金+项目奖金+科研补贴+弹性工作",
        "赵晓雨": "性别：女，年龄：27，手机号：13488776655，邮箱：zxy@example.com，工作：产品经理，公司：XX科技有限公司，部门：产品部，岗位：C端产品经理，职级：中级，薪资：20000元/月，入职时间：2021-05-22，学历：本科（产品设计），技能：Axure、Figma、用户调研、PRD撰写，直属领导：孙产品总监，团队：用户增长组，福利：五险一金+产品提成+年度旅游+下午茶",
        "周宇航": "性别：男，年龄：26，手机号：13377665544，邮箱：zyh@example.com，工作：运营专员，公司：XX科技有限公司，部门：运营部，岗位：内容运营，职级：初级，薪资：11000元/月，入职时间：2023-02-18，学历：本科（汉语言文学），技能：文案撰写、公众号运营、数据分析（基础）、活动策划，直属领导：吴运营主管，团队：内容运营组，福利：五险一金+绩效奖金+节日礼品+带薪病假",
        "吴佳琪": "性别：女，年龄：30，手机号：13266554433，邮箱：wjq@example.com，工作：运营经理，公司：XX科技有限公司，部门：运营部，岗位：用户运营，职级：高级，薪资：22000元/月，入职时间：2020-04-30，学历：本科（市场营销），技能：用户分层、社群运营、活动复盘、CRM系统操作，直属领导：马运营总监，团队：用户运营组，福利：五险一金+管理津贴+年度体检+团队建设基金",
        "钱浩宇": "性别：男，年龄：35，手机号：13199887766，邮箱：qhy@example.com，工作：技术架构师，公司：XX科技有限公司，部门：架构部，岗位：高级架构师，职级：专家，薪资：40000元/月，入职时间：2018-03-12，学历：硕士（计算机科学），技能：系统架构设计、分布式系统、高并发处理、云原生，直属领导：CTO，团队：架构组，福利：五险一金+股票期权+高额年终奖+弹性工作制",
        "孙雅琳": "性别：女，年龄：28，手机号：13088776655，邮箱：syl@example.com，工作：财务专员，公司：XX科技有限公司，部门：财务部，岗位：成本会计，职级：中级，薪资：16000元/月，入职时间：2021-08-05，学历：本科（会计学），技能：财务软件、成本核算、税务申报、报表分析，直属领导：财务经理，团队：财务组，福利：五险一金+年度体检+节日福利+周末双休",
        "郑子昂": "性别：男，年龄：27，手机号：12977665544，邮箱：zza@example.com，工作：UI设计师，公司：XX科技有限公司，部门：设计部，岗位：高级UI设计师，职级：高级，薪资：18000元/月，入职时间：2020-12-15，学历：本科（视觉传达设计），技能：Figma、PS、AI、交互设计、用户体验，直属领导：设计总监，团队：设计组，福利：五险一金+创意奖金+设计培训+设备补贴",
        "冯小梦": "性别：女，年龄：26，手机号：12866554433，邮箱：fxm@example.com，工作：人力资源专员，公司：XX科技有限公司，部门：人力资源部，岗位：招聘专员，职级：中级，薪资：14000元/月，入职时间：2022-01-10，学历：本科（人力资源管理），技能：招聘流程、人才测评、面试技巧、员工关系，直属领导：HR总监，团队：招聘组，福利：五险一金+团建活动+生日福利+带薪年假12天",
    }
    return dic.get(name, "未找到该人员信息")

TOOLS = [search_docs]

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
).bind_tools(TOOLS)


In [48]:
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
).bind_tools(tools)

In [52]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(TOOLS)
graph_builder.add_node("tools", tool_node)

def route_tools(state: State):
    """
    Use in the conditional_edge to route to the ToolNode if the last message
    has tool calls. Otherwise, route to the end.
    """
    if isinstance(state, list):
        ai_message = state[-1]
    elif messages := state.get("messages", []):
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    return END

graph_builder.add_conditional_edges(
    "chatbot",
    route_tools,
    {
        "tools": "tools",
        END: END,
    },
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile()

## 可视化图

In [53]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [54]:

def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break

        stream_graph_updates(user_input)
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

Assistant: 你好！有什么我可以帮助你的吗？
Assistant: 我无法提供个人隐私信息，如需联系王小明，请通过正当途径获取其联系方式。
Assistant: Hello! How can I assist you today?
Goodbye!


## 添加记忆